#### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behaviour with logging, analytics and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retires, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII Detection

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

#### Summarization Middleware

Automatically summarize conversation history when approaching toke limits, preserving recent messages when compressing older context. Summarization is useful for the following:

* Long-running conversations that exceed context windows
* Multi-turn dialogues with extensive history
* Applications where preserving full conoversation context matters

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

agent = create_agent(
    model = ChatGroq(model="qwen/qwen3.6-27b"),
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=ChatGroq(model="qwen/qwen3.6-27b"),
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [6]:
### Run with thread id

config={"configurable": {"thread_id":"test-1"}}

In [7]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='b0dea66a-500e-4f48-b5b9-c26140c57b94'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Task:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely. "2 + 2 equals 4."\n5.  **Self-Correction/Verification:** The calculation is straightforward and universally accepted. No ambiguity. Response is accurate.\n6.  **Output Generation:** Provide the direct answer.✅\n</think>\n\n2 + 2 equals 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 17, 'total_tokens': 158, 'completion_time': 0.268688455, 'completion_tokens_details': None, 'prompt_time': 0.000920015, 'prompt_tokens_details': None, 'queue_time': 0.049590604, 'total_time': 0.26960847}, 'model_name

## Based on token size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Hotel 1
    2. Hotel 2
    3. Hotel 3
    """

agent = create_agent(
    model = ChatGroq(model="qwen/qwen3.6-27b"),
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = ChatGroq(model="qwen/qwen3.6-27b"),
            trigger=("tokens", 500),
            keep=("tokens", 200)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token Counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars//4

In [9]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~54 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='3817daac-a626-4ef6-b583-ef0c4c11fc7e'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to find hotels in Paris.\n2.  **Identify Available Tools**: `search_hotels` takes a `city` parameter (string).\n3.  **Map Intent to Tool**: Call `search_hotels` with `city: "Paris"`.\n4.  **Execute Tool Call**: Generate the function call.✅\n5.  **Handle Response**: Wait for the tool output, then format it nicely for the user. (I will just output the tool call now).👇🏼\n- Check parameters: `city` is required. User provided "Paris".\n- Construct JSON: `{"city": "Paris"}`.\n- Proceed. \n- Self-Correction/Verification during thought: The tool description says "returns long response to use more tokens". I will just call it. No extra constraints. All good.✅\n- Output matches tool call format.👇🏼\n', 'tool_ca

#### Human in the Loop Middleware

Pause agent execution for human approval, editing, or rejection of too calls before they execute. Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory
* Long-running conversations where human feedback guides the agent

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [11]:
agent = create_agent(
    model = ChatGroq(model="qwen/qwen3.6-27b"),
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ['approve', 'edit', 'reject']
                },
                "read_email_tool": False
            }
        )
    ]
)

In [13]:
config = {"configurable": {"thread_id": "test-approve"}}

# step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you'")]},
    config
)

In [16]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='e8c0ad98-d8b6-437b-a7eb-f2269c0c19c4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: `john@test.com`\n   - Subject: `hello`\n   - Body: `How are you`\n2.  **Identify Tool**: `send_email_tool` matches the requirement.\n3.  **Check Parameters**:\n   - `recipient`: "john@test.com"\n   - `subject`: "hello"\n   - `body`: "How are you"\n   All required parameters are provided.\n4.  **Execute Tool Call**: Call `send_email_tool` with the specified parameters.✅\n5.  **Formulate Response**: After tool execution, provide a confirmation message. (Wait for tool output first, but I\'ll just generate the tool call now).👇🏼\n   `send_email_tool(recipient="john@test.com", subject="hello", body="How are you")`📝\n   All good. Proceed. \n  

In [18]:
from langgraph.types import Command
# Step 2: Approve

if "__interrupt__" in result:
    print("Paused!! Approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type":  "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused!! Approving
Result: The email has been successfully sent to john@test.com with the subject 'hello'.


In [19]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='e8c0ad98-d8b6-437b-a7eb-f2269c0c19c4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: `john@test.com`\n   - Subject: `hello`\n   - Body: `How are you`\n2.  **Identify Tool**: `send_email_tool` matches the requirement.\n3.  **Check Parameters**:\n   - `recipient`: "john@test.com"\n   - `subject`: "hello"\n   - `body`: "How are you"\n   All required parameters are provided.\n4.  **Execute Tool Call**: Call `send_email_tool` with the specified parameters.✅\n5.  **Formulate Response**: After tool execution, provide a confirmation message. (Wait for tool output first, but I\'ll just generate the tool call now).👇🏼\n   `send_email_tool(recipient="john@test.com", subject="hello", body="How are you")`📝\n   All good. Proceed. \n  

# Reject

In [20]:
agent = create_agent(
    model = ChatGroq(model="qwen/qwen3.6-27b"),
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ['approve', 'edit', 'reject']
                },
                "read_email_tool": False
            }
        )
    ]
)

In [21]:
config = {"configurable": {"thread_id": "test-reject"}}

# step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you'")]},
    config
)

In [22]:
from langgraph.types import Command
# Step 2: Approve

if "__interrupt__" in result:
    print("Paused!! Approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type":  "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused!! Approving
Result: 


In [23]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='2aa3c60a-f4f5-420f-b5d7-aa770a54d68f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: john@test.com\n   - Subject: \'hello\'\n   - Body: \'How are you\'\n2.  **Identify Available Tools**: `send_email_tool` matches the requirement.\n   - Parameters: `recipient` (string), `subject` (string), `body` (string). All required.\n3.  **Map Input to Tool Parameters**:\n   - `recipient`: "john@test.com"\n   - `subject`: "hello"\n   - `body`: "How are you"\n4.  **Execute Tool Call**: Construct the function call.\n   - `send_email_tool(recipient="john@test.com", subject="hello", body="How are you")`\n5.  **Output Generation**: Return the tool call.✅\n', 'tool_calls': [{'id': 'qn5k32yt4', 'function': {'arguments': '{"body":"How are yo